# Context windows & the "lost in the middle" problem

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 8 §8.1, §8.5 · type: concept-lab

**The promise:** by the end you can explain why long context costs more (attention is ~quadratic) and why a fact's *position* in the prompt changes whether the model actually uses it.

Offline by default: a toy attention matrix plus a mocked, position-dependent needle test. No API key needed.

## 🧠 Why this matters

The transformer's key move is **attention**: as the model processes a sequence, every token gets to "look at" every earlier token and decide which ones matter for predicting what comes next. That is enormously powerful — but comparing every position against every other position makes cost grow *roughly quadratically* with sequence length. The window is finite for a reason, and long contexts leak out to you as **price and latency**.

There's a subtler trap. Models attend *unevenly* across long inputs: information at the **beginning** and **end** is used reliably, while facts buried in the **middle** of a huge context are sometimes effectively ignored. This is the *lost in the middle* effect, and it means "the fact is in the context window" is necessary but **not** sufficient for the model to use it. See §8.1 and §8.5.

## Objectives & prereqs

**By the end you can:**
- Read a toy attention matrix and see "every position attends to every earlier position."
- Build a needle-in-a-haystack test and observe the position-dependent recovery pattern.
- Justify *curated* context over *voluminous* context, and place key facts where they're used.

**Prereqs:** [`08-01-tokens-and-the-bill.ipynb`](08-01-tokens-and-the-bill.ipynb). Chapter 8 read.

**Packages:** standard library only in `MOCK=1`. The optional live needle test (`MOCK=0`) uses `anthropic` (in `requirements.txt`); its token cost scales with the filler length and is flagged at that cell.

In [ ]:
# Setup — imports, env, MOCK switch, fixed seed.
import os
import random

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default): toy attention + a canned, position-dependent needle result.
#   Fully offline, free, deterministic.
# MOCK=0: optionally run the needle test against a real long-context model.
#   Token cost scales with the filler size (flagged at that cell).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(8)  # determinism for the generated filler and toy weights

MODEL = "claude-opus-4-8"  # the book's default model (used only on the live path)

print(f"MOCK mode: {MOCK}  (toy attention + canned needle test, offline)" if MOCK
      else "LIVE mode: needle test will hit a real long-context model (token cost scales with filler)")

## A toy attention matrix

We won't build a real transformer. Instead, a *tiny illustration*: a short sequence and a weight matrix where row *i* shows how much token *i* attends to each earlier token *j*. The shape is what matters — it's **lower-triangular** (a token can only look *backward*), and every row's weights sum to 1.

This is a cartoon of attention, not the math from §8.1 — but it makes "every position attends to every earlier position" something you can see.

In [ ]:
tokens = ["The", "capital", "of", "France", "is"]
n = len(tokens)


def toy_attention(tokens: list[str]) -> list[list[float]]:
    """Illustrative causal attention weights (lower-triangular, rows sum to 1).

    Each token attends only to itself and earlier tokens. Weights here are made
    up for teaching; a real model learns them. Deterministic via the seed above.
    """
    n = len(tokens)
    rows = []
    for i in range(n):
        raw = [random.random() for _ in range(i + 1)] + [0.0] * (n - i - 1)
        total = sum(raw[: i + 1])
        rows.append([(w / total if j <= i else 0.0) for j, w in enumerate(raw)])
    return rows


weights = toy_attention(tokens)

header = "attends->  " + " ".join(f"{t[:6]:>6s}" for t in tokens)
print(header)
for i, row in enumerate(weights):
    cells = " ".join(f"{w:6.2f}" if w else "     ." for w in row)
    print(f"{tokens[i][:9]:>9s}  {cells}")
print("\nRead a row: how much that token looks back at each earlier token.")
print("Upper triangle is empty — no token can attend to the future.")

**What you just saw.** Predicting the token after "...France is" lets the last position attend back over *all* prior tokens and weight "capital" and "France" heavily. The cost hint is right there in the triangle: a sequence of length *N* has on the order of *N²/2* of these pairwise weights. Double the context, roughly quadruple this work — that's why long contexts cost and wait more.

In [ ]:
# The quadratic, made concrete: pairwise comparisons grow with the square of length.
print(f"{'tokens':>8s}  {'pairwise comparisons (~n^2/2)':>30s}")
for length in [10, 100, 1_000, 10_000, 100_000]:
    print(f"{length:>8,d}  {length * length // 2:>30,d}")
print("\nThis is why 'it fits in the window' is not the same as 'it's cheap'.")

## A needle in a haystack

Now the famous test. We plant one fact (the "needle") at a chosen position inside a long block of filler (the "haystack"), then ask the model to recover it. We'll run the *same* needle at the **start**, **middle**, and **end** and compare recovery.

The filler is generated by the next cell — it is **not** committed to the repo (no binary blob), and it's deterministic via the seed.

In [ ]:
FILLER_SENTENCES = [
    "The quarterly review covered staffing, tooling, and roadmap risks.",
    "Most teams reported steady progress against their objectives.",
    "Infrastructure spend was flat compared with the previous period.",
    "A few minor incidents were resolved without customer impact.",
    "Hiring continued at a measured pace across the org.",
]
NEEDLE = "The launch code for the orbital relay is ZEPHYR-7."
QUESTION = "What is the launch code for the orbital relay?"
ANSWER = "ZEPHYR-7"


def build_haystack(n_sentences: int, needle: str, position: str) -> str:
    """Generate filler text with the needle planted at start | middle | end.

    Deterministic (seeded above). Built at runtime, never committed.
    """
    filler = [random.choice(FILLER_SENTENCES) for _ in range(n_sentences)]
    if position == "start":
        idx = 0
    elif position == "end":
        idx = len(filler)
    else:  # middle
        idx = len(filler) // 2
    filler.insert(idx, needle)
    return " ".join(filler)


demo = build_haystack(6, NEEDLE, "middle")
print(f"haystack length: {len(demo)} chars; needle planted in the MIDDLE.\n")
print(demo)

## 🔮 Predict: which position is recovered most reliably?

We'll plant the needle at the **start**, **middle**, and **end** of a long haystack and ask the question each time. Before you run the next cell, **predict**: which position(s) will the model recover reliably, and which will it tend to miss?

(If you've read §8.5, you know the shape of the answer — say it out loud anyway.)

In [ ]:
def needle_test(position: str, n_sentences: int = 200) -> str:
    """Ask the model to recover the needle planted at `position`.

    MOCK=1: a canned, position-dependent result that reproduces the documented
        'lost in the middle' pattern (start/end recovered, middle often missed).
        Deterministic and offline.
    MOCK=0: a real call against a long-context model. WARNING: token cost scales
        with `n_sentences` (a 200-sentence haystack is a sizable prompt).
    """
    haystack = build_haystack(n_sentences, NEEDLE, position)
    prompt = f"{haystack}\n\nQuestion: {QUESTION}\nAnswer with just the code."

    if MOCK:
        # Canned pattern: facts at the edges are used reliably; a mid-context fact
        # is frequently overlooked even though it IS in the window. Illustrative.
        recovered = {"start": True, "end": True, "middle": False}[position]
        return ANSWER if recovered else "I couldn't find that in the text."

    import anthropic  # imported lazily so MOCK users need no SDK/key

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    resp = client.messages.create(
        model=MODEL,
        max_tokens=32,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in resp.content if b.type == "text").strip()


print(f"Needle: {NEEDLE!r}")
print(f"Question: {QUESTION!r}\n")
for pos in ("start", "middle", "end"):
    result = needle_test(pos)
    hit = "FOUND   " if ANSWER in result else "MISSED  "
    print(f"  needle at {pos:>6s}:  {hit} -> {result!r}")

**What you just saw.** The needle at the **start** and **end** is recovered; the one in the **middle** is missed — even though, in every case, the fact was sitting right there in the context window. "It fits" did not mean "it's used." (In `MOCK` this pattern is canned to be crisp; on a real model the effect is statistical — flakier, but the same shape, and it worsens as the haystack grows.)

## 🎯 Senior lens

A senior doesn't fight "lost in the middle" by buying a bigger window — they design around it. Two habits:

1. **Curate, don't dump.** Retrieve the *relevant ten chunks* (Chapter 13) instead of stuffing the whole wiki into the prompt. Less context that's all relevant beats a sea of context the model wades through.
2. **Place facts where they're read.** Put the most important material at the **start or end** of a long prompt, never buried in the middle. System instructions and the user's actual question are natural anchors at the edges.

The underlying mindset: **every token in the prompt must earn its place.** Each one costs money, adds latency, and dilutes attention on the tokens that matter.

## ⚠️ Pitfall: "it fits in the window" is not a retrieval strategy

Teams discover long contexts, dump entire knowledge bases into the prompt, and ship. Then quality quietly degrades: cost and latency triple (the quadratic from earlier), *and* the model overlooks mid-context facts. The window growing to hundreds of thousands of tokens did not repeal the geometry — it just gave you more rope. Curated, relevant context beats voluminous context every time.

## Recap

- **Attention** lets every token look at every earlier token; cost grows ~**quadratically** with length.
- That cost reaches you as **price and latency**, which is why windows are finite and long prompts are expensive.
- **Lost in the middle:** facts at the start/end are used reliably; mid-context facts are often ignored.
- "In the window" ≠ "used by the model" — *position* matters.
- **Curate context** and place key facts at the edges; make every token earn its place.

## Exercises

Each task *changes* something; predict before you run.

1. **Scale the haystack.** Re-run `needle_test` with `n_sentences` of 20, 200, and 2000. In `MOCK` the pattern is fixed — but predict how a *real* model's middle-recovery would change as the haystack grows, and why.
2. **Two needles.** Plant one fact at the start and a *different* one in the middle; ask for both. Predict which you'd reliably get back.
3. **Cost estimate.** Reuse `anthropic_token_count` from notebook 08-01 to measure the token size of a 200-sentence haystack, and estimate how latency/price scale vs. a 20-sentence one.
4. **(Live, optional)** With `COMPANION_MOCK=0`, a key, and a *small* `n_sentences`, run the real needle test a few times per position. Is the middle genuinely flakier than the edges?

In [ ]:
# Exercise scratch space — your code here.


In [ ]:
# Exercise scratch space — your code here.


## Next

- **This wraps Part III's substrate labs (Ch 8).** Next chapter: [`../09-inference-sampling-control/`](../09-inference-sampling-control/) — from what the model *is* to how you *control* its output (sampling knobs, determinism, streaming, latency).
- **Book:** §8.1 (attention) and §8.5 (context windows, lost in the middle).
- **Where this leads:** the "curate, don't dump" rule is the whole motivation for the RAG pipeline ([`blueprints/rag-pipeline/`](../../blueprints/rag-pipeline/)) and the capstone's `rag/` module (Chapter 13). This chapter gives you the reason; those build the machinery.